# 成员 B：出发延误预测演示

本 Notebook 读取成员 A 的计划特征，展示 Logistic Regression 与 Random Forest 的对比、Precision / Recall / F1，以及风险等级。请先在 `B_models` 目录运行 `python scripts/train_models.py`。

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

root = Path.cwd()
if not (root / "config.json").exists():
    root = root.parent
assert (root / "config.json").exists(), "请从 B_models 或 notebooks 目录启动"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.predict import artifacts_exist, load_artifacts, predict_schedule
reports = root / "reports"
figures = reports / "figures"
print("项目目录:", root)
print("模型已训练:" , artifacts_exist())

## 问题与数据口径

仅用计划信息预测登机口出发延误是否达到 15 分钟。训练 / 验证 / 测试按当地航班日期划分，不打乱。类别编码和标准化都在训练 Pipeline 内完成。

In [ ]:
valid = pd.read_csv(reports / "metrics_validation.csv")
test = pd.read_csv(reports / "metrics_test.csv")
display(Markdown("### 验证集（用于选模型与阈值）"))
display(valid)
display(Markdown("### 测试集（最终评价）"))
display(test)

In [ ]:
for name in ["validation_prf.png", "validation_roc_pr.png", "test_roc_pr.png", "confusion_validation_selected.png"]:
    path = figures / name
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

## 选定模型与风险等级

风险切分来自验证集预测概率的 40% / 80% 分位数，再冻结到测试集。High 并不保证一定延误，只表示相对更高风险。

In [ ]:
selection = json.loads((root / "models" / "selection.json").read_text(encoding="utf-8"))
policy = json.loads((root / "models" / "risk_policy.json").read_text(encoding="utf-8"))
print(json.dumps(selection, ensure_ascii=False, indent=2))
display(pd.read_csv(reports / "risk_bands_validation.csv"))
display(pd.read_csv(reports / "risk_bands_test.csv"))
for name in ["risk_validation.png", "risk_test.png"]:
    path = figures / name
    if path.exists():
        display(Image(filename=str(path)))

## 单次计划航班预测

下面只使用计划字段。修改航空公司、机场、星期和计划时刻后重新运行即可。

In [ ]:
artifacts = load_artifacts()
result = predict_schedule(
    artifacts,
    airline="AA",
    origin="DFW",
    dest="PHX",
    day_of_week=3,
    crs_dep_time=1400,
    distance=868,
    crs_elapsed_time=160,
)
display(pd.DataFrame([result]))

In [ ]:
pred = pd.read_csv(root / "data" / "predictions" / "test.csv")
display(Markdown("### 测试集预测抽样"))
display(pred.sample(15, random_state=42)[
    ["FlightDate", "Airline", "FlightNumber", "Origin", "Dest", "CRSDepTime",
     "delay_probability", "predicted_delayed15", "risk_level", "TargetDelayed15"]
])